In [0]:

df=spark.read.csv("/Volumes/workspace/default/databricks_training/Day5_pyspark_practice_Big_Sales.csv",header=True,inferSchema=True)
df.display();

In [0]:
from pyspark.sql.functions import *

df.select(col("Item_Identifier").alias("item_id")).display()

In [0]:
df.filter(col("Item_Fat_Content")=="Regular").display()

In [0]:
df.filter((col("Item_Type")=="Soft Drinks") & (col("Item_Weight")<10)).display()

In [0]:
df.filter((col("Outlet_Location_Type").isin("Tier1","Tier2")) & col("Outlet_Size").isNull()).display()

In [0]:
df.filter(col("Outlet_Size").isNotNull()).display()

withColumnRenamed

In [0]:
df.withColumnRenamed("Item_Weight","Item_Wt").display()

withColumn

In [0]:
df.withColumn("flag",lit("new")).display()

regexp_replace()-used to replace the text

In [0]:
# item fat content low fat as lf and regular as reg
df.withColumn("Item_Fat_Content", regexp_replace(regexp_replace(col("Item_Fat_Content"), "Regular", "Reg"), "Low Fat", "LF")).display()

In [0]:
# to replace outlet size whose value is null to na with also above conditions
from pyspark.sql.functions import col, regexp_replace, when

df = df.withColumn(
    "Item_Fat_Content",
    regexp_replace(
        regexp_replace(col("Item_Fat_Content"), "Regular", "Reg"),
        "Low Fat", "LF"
    )
).withColumn(
    "Outlet_Size",
    when(col("Outlet_Size").isNull(), "NA")
    .otherwise(col("Outlet_Size"))
)

df.display()
# regexp_replace() works on strings, not NULL values

sorting


In [0]:
# sort the column Item_Weight in descending order

In [0]:
df.sort(col("Item_Weight").desc()).display()

In [0]:
# sort the column Item_Visibility in ascending order
df.sort(df.Item_Visibility.asc()).display()

In [0]:
df.sort(
    col("Item_Weight").asc(),
    col("Item_Visibility").asc()
).display()

In [0]:
df.limit(5).display()

### ddl and structType()-for changing the one data type to another where we don't want the datatype which is defined by the spark

## ddl

In [0]:
df.printSchema()

###  Now i want to change the item weight to a string type which is just a hypothetical situation then by using ddl it is as follows


In [0]:
my_schema='''
  Item_Identifier string,
  Item_Weight string,
  Item_Fat_Content string,
  Item_Visibility double,
  Item_Type string,
  Item_MRP double,
  Outlet_Identifier string,
  Outlet_Establishment_Year int,
  Outlet_Size string,
  Outlet_Location_Type string,
  Outlet_Type string,
  Item_Outlet_Sales double
  '''

In [0]:
df=spark.read.format('csv')\
    .schema(my_schema).option('header',True)\
        .load('/Volumes/workspace/default/databricks_training/Day5_pyspark_practice_Big_Sales.csv')

In [0]:
df.printSchema()

### StructType()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
my_struct_schema=StructType([
    StructField('Item_Identifier',StringType(),True),
    StructField('Item_Weight',StringType(),True),
    StructField('Item_Fat_Content',StringType(),True),
    StructField('Item_Visibility',DoubleType(),True),
    StructField('Item_Type',StringType(),True),
    StructField('Item_MRP',DoubleType(),True),
    StructField('Outlet_Identifier',StringType(),True),
    StructField('Outlet_Establishment_Year',IntegerType(),True),
    StructField('Outlet_Size',StringType(),True),
    StructField('Outlet_Location_Type',StringType(),True),
    StructField('Outlet_Type',StringType(),True),
    StructField('Item_Outlet_Sales',DoubleType(),True)
    ])

df=spark.read.format('csv')\
    .schema(my_struct_schema).option('header',True)\
        .load('/Volumes/workspace/default/databricks_training/Day5_pyspark_practice_Big_Sales.csv')
df.printSchema()

In [0]:
df.display()

# Transformations

## select


There are two ways of displaying with the select statement one is directly using the column names inside select and the other is using col() and defined as follows

In [0]:
df.select("Item_Identifier","Item_Weight").display()

In [0]:
df.select(col("Item_Fat_Content"),col("Item_Identifier")).display()

## Alias

In [0]:
df.select(col('Item_Identifier').alias('Item_ID')).display()

## UNION AND UNION BYNAME

In [0]:
data1=[('1','kad'),
       ('2','sid')]
schema1='id STRING,name STRING'

data2=[('3','rahul'),('4','jas')]
schema2='id STRING,name STRING'

df1=spark.createDataFrame(data1,schema1)
df2=spark.createDataFrame(data2,schema2)

In [0]:
df1.display()

In [0]:
df2.display()

In [0]:
df1.union(df2).display()

In [0]:
data1=[('kad','1'),
       ('sid','2')]
schema1='name STRING,id STRING'
df1=spark.createDataFrame(data1,schema1)
df1.display()

In [0]:
df1.union(df2).display()
df1.unionByName(df2).display()


# STRING FUNCTIONS


## 1.INIT CAP

In [0]:
from pyspark.sql.functions import *
df.select(initcap('Item_Type')).display()

### lower()

In [0]:
df.select(lower('Item_Type')).display()

### upper()

In [0]:
df.select(upper('Item_Type').alias('Upper_Item_Type')).display()

# JOINS

In [0]:
dataj1 = [
    ('1', 'gaur', 'd01'),
    ('2', 'kit', 'd02'),
    ('3', 'sam', 'd03'),
    ('4', 'tim', 'd03'),
    ('5', 'aman', 'd05'),
    ('6', 'jim', 'd06')
]

schemaj1 = 'emp_id STRING, emp_name STRING, dept_id STRING'

df1 = spark.createDataFrame(dataj1, schemaj1)

dataj2 = [
    ('d01', 'HR'),
    ('d02', 'Marketing'),
    ('d03', 'Accounts'),
    ('d04', 'IT'),
    ('d05', 'Finance')
]

schemaj2 = 'dept_id STRING, department STRING'
df2=spark.createDataFrame(dataj2,schemaj2)

In [0]:
df1.display()
df2.display()

## inner join

In [0]:
df1.join(df2,df1['dept_id']==df2['dept_id'],'inner').display()

## left join

In [0]:
df1.join(df2,df1['dept_id']==df2['dept_id'],'left').display()

## right join

In [0]:
df1.join(df2,df1['dept_id']==df2['dept_id'],'right').display()

## ANTI JOIN

An anti join in PySpark is used to get records from one DataFrame that do NOT have a match in another DataFrame.

In [0]:
df2.join(df1,df1['dept_id']==df2['dept_id'],'anti').display()

# window functions

## row_number()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

df.withColumn('rowcol',row_number().over(Window.orderBy('Item_Identifier'))).display()